In [4]:
import os

print('原始路径：' + os.getcwd())
os.chdir('/workspace/')
print('新路径：' + os.getcwd())


原始路径：/
新路径：/workspace


In [5]:
from unsloth import FastLanguageModel

max_seq_length = 1024
model_name = 'unsloth/DeepSeek-R1-Distill-Qwen-7B-unsloth-bnb-4bit'

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True
)



🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
INFO 09-28 06:34:58 [__init__.py:216] Automatically detected platform cuda.
ERROR 09-28 06:34:59 [fa_utils.py:57] Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.8.5: Fast Qwen2 patching. Transformers: 4.55.4. vLLM: 0.10.2.
   \\   /|    Tesla V100-SXM2-32GB. Num GPUs = 1. Max memory: 31.739 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 7.0. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [15]:
inference_prompt = """以下是一条描述任务的指令，并配有一个提供进一步上下文的输入。
请撰写一份恰当的回复，以完成该请求。
在回答之前，请仔细思考该问题，并构建一个分步的思考过程，以确保回应的逻辑严谨和内容准确。


### Instruction:
你是一位医学专家，在临床推理、诊断学和治疗规划方面拥有深厚的专业知识。
请回答以下医学问题。

### Question:
{}

### Response:
<think>{}
"""

FastLanguageModel.for_inference(model)

question = '男，28岁，程序员，最近一周每天工作到半夜，感觉头晕、脖子疼，有时候还恶心。'
formatedPrompt = inference_prompt.format(question, '')
print(formatedPrompt)

inputs = tokenizer([formatedPrompt], return_tensors='pt').to('cuda')
attention_mask = inputs.input_ids.ne(tokenizer.pad_token_id).to('cuda')

outputs = model.generate(
    input_ids = inputs.input_ids,
    attention_mask = inputs.attention_mask,
    max_new_tokens = 1000,
    use_cache = True,
)


以下是一条描述任务的指令，并配有一个提供进一步上下文的输入。
请撰写一份恰当的回复，以完成该请求。
在回答之前，请仔细思考该问题，并构建一个分步的思考过程，以确保回应的逻辑严谨和内容准确。


### Instruction:
你是一位医学专家，在临床推理、诊断学和治疗规划方面拥有深厚的专业知识。
请回答以下医学问题。

### Question:
男，28岁，程序员，最近一周每天工作到半夜，感觉头晕、脖子疼，有时候还恶心。

### Response:
<think>



In [16]:
print(inputs)
print(outputs)


response = tokenizer.batch_decode(outputs, skip_special_tokens=True)
#print(response)
splitedResponse = response[0].split('### Response:')
print(response[0].split('### Response:')[1])

{'input_ids': tensor([[151646,  87752,  99639,  38989,  53481,  88802,   9370, 109504,  90395,
          54387, 104133,  99553, 100642, 102285,  16744,   9370,  31196,   8997,
          14880, 110479, 104191, 112449,   9370, 104787,   3837,  23031,  60548,
          75882,  34859,   8997,  18493, 102104, 101056,  37945, 104857, 104107,
          75882,  86119,  90395, 104004,  46944,  17177,  64682,   9370, 104107,
         100178,   3837,  23031, 103944, 104493,   9370, 104913, 108487,  33108,
          43815, 102188,   1773,   1406,  14374,  29051,    510,  56568, 109182,
         104316, 101057,  96050, 104595, 113272,   5373, 105262,  47764,  33108,
         101899, 100367,  99522, 103926, 103524, 106289, 100032,   8997,  14880,
         102104,  87752, 104316,  86119,   3407,  14374,  15846,    510,  70108,
           3837,     17,     23,  92015,   3837, 118552,   3837, 104044, 105309,
         101922,  99257,  26939, 111001,   3837, 100681, 116280,   5373, 107966,
         10090

In [8]:
# 模型训练的 Prompt 模板
train_prompt = """以下是一条描述任务的指令，并配有一个提供进一步上下文的输入。
请撰写一份恰当的回复，以完成该请求。
在回答之前，请仔细思考该问题，并构建一个分步的思考过程，以确保回应的逻辑严谨和内容准确。


### Instruction:
你是一位医学专家，在临床推理、诊断学和治疗规划方面拥有深厚的专业知识。
请回答以下医学问题。

### Question:
{}

### Response:
<think>
{}
</think>
{}
"""

EOS_TOKEN = tokenizer.eos_token # 添加 EOS Token

def formatting_prompts_func(examples):
    inputs = examples["Question"]
    cots = examples["Complex_CoT"]
    outputs = examples["Response"]
    texts = []
    for input, cot, output in zip(inputs, cots, outputs):
        # 将 EOS Token 添加到样本最后
        text = train_prompt.format(input, cot, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }
pass

from datasets import load_dataset

dataset = load_dataset("FreedomIntelligence/medical-o1-reasoning-SFT", "zh", split = "train")
dataset = dataset.map(formatting_prompts_func, batched = True,)


In [6]:
from IPython.display import display, Markdown

display(Markdown(dataset[0]["text"])) 

以下是一条描述任务的指令，并配有一个提供进一步上下文的输入。
请撰写一份恰当的回复，以完成该请求。
在回答之前，请仔细思考该问题，并构建一个分步的思考过程，以确保回应的逻辑严谨和内容准确。


### Instruction:
你是一位医学专家，在临床推理、诊断学和治疗规划方面拥有深厚的专业知识。
请回答以下医学问题。

### Question:
根据描述，一个1岁的孩子在夏季头皮出现多处小结节，长期不愈合，且现在疮大如梅，溃破流脓，口不收敛，头皮下有空洞，患处皮肤增厚。这种病症在中医中诊断为什么病？

### Response:
<think>
这个小孩子在夏天头皮上长了些小结节，一直都没好，后来变成了脓包，流了好多脓。想想夏天那么热，可能和湿热有关。才一岁的小孩，免疫力本来就不强，夏天的湿热没准就侵袭了身体。

用中医的角度来看，出现小结节、再加上长期不愈合，这些症状让我想到了头疮。小孩子最容易得这些皮肤病，主要因为湿热在体表郁结。

但再看看，头皮下还有空洞，这可能不止是简单的头疮。看起来病情挺严重的，也许是脓肿没治好。这样的情况中医中有时候叫做禿疮或者湿疮，也可能是另一种情况。

等一下，头皮上的空洞和皮肤增厚更像是疾病已经深入到头皮下，这是不是说明有可能是流注或瘰疬？这些名字常描述头部或颈部的严重感染，特别是有化脓不愈合，又形成通道或空洞的情况。

仔细想想，我怎么感觉这些症状更贴近瘰疬的表现？尤其考虑到孩子的年纪和夏天发生的季节性因素，湿热可能是主因，但可能也有火毒或者痰湿造成的滞留。

回到基本的症状描述上看，这种长期不愈合又复杂的状况，如果结合中医更偏重的病名，是不是有可能是涉及更深层次的感染？

再考虑一下，这应该不是单纯的瘰疬，得仔细分析头皮增厚并出现空洞这样的严重症状。中医里头，这样的表现可能更符合‘蚀疮’或‘头疽’。这些病名通常描述头部严重感染后的溃烂和组织坏死。

看看季节和孩子的体质，夏天又湿又热，外邪很容易侵入头部，对孩子这么弱的免疫系统简直就是挑战。头疽这个病名听起来真是切合，因为它描述的感染严重，溃烂到出现空洞。

不过，仔细琢磨后发现，还有个病名似乎更为合适，叫做‘蝼蛄疖’，这病在中医里专指像这种严重感染并伴有深部空洞的情况。它也涵盖了化脓和皮肤增厚这些症状。

哦，该不会是夏季湿热，导致湿毒入侵，孩子的体质不能御，其病情发展成这样的感染？综合分析后我觉得‘蝼蛄疖’这个病名真是相当符合。
</think>
从中医的角度来看，你所描述的症状符合“蝼蛄疖”的病症。这种病症通常发生在头皮，表现为多处结节，溃破流脓，形成空洞，患处皮肤增厚且长期不愈合。湿热较重的夏季更容易导致这种病症的发展，特别是在免疫力较弱的儿童身上。建议结合中医的清热解毒、祛湿消肿的治疗方法进行处理，并配合专业的医疗建议进行详细诊断和治疗。
<｜end▁of▁sentence｜>

In [6]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing="unsloth",
    random_state=1432,
    use_rslora=False,
    loftq_config=None,
)

print(model)

Unsloth 2025.8.5 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(152064, 3584, padding_idx=151654)
        (layers): ModuleList(
          (0-3): 4 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=3584, out_features=3584, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3584, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=3584, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(
 

In [9]:
from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = 'text',
    max_seq_length = 1024,
    packing = False,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 32,
        warmup_steps = 5,
        max_steps = 40,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 1432,
        output_dir = "outputs-7B",
        report_to = "none",
    )
)


Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


In [10]:
import torch

trainer_stats = trainer.train()

print(trainer_stats)

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 20,171 | Num Epochs = 1 | Total steps = 40
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 32
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 32 x 1) = 64
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,entropy
1,3.122300,0
2,3.029400,No Log
3,3.039900,No Log
4,3.014500,No Log
5,2.843000,No Log
6,2.804500,No Log
7,2.738100,No Log
8,2.690100,No Log
9,2.600100,No Log
10,2.480100,No Log


TrainOutput(global_step=40, training_loss=2.275097021460533, metrics={'train_runtime': 1559.9194, 'train_samples_per_second': 1.641, 'train_steps_per_second': 0.026, 'total_flos': 7.710383134295654e+16, 'train_loss': 2.275097021460533, 'epoch': 0.12690858615903233})


In [11]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla V100-SXM2-32GB. Max memory = 31.739 GB.
11.1 GB of memory reserved.


In [12]:
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

Peak reserved memory = 11.1 GB.
Peak reserved memory for training = 0.0 GB.
Peak reserved memory % of max memory = 34.973 %.
Peak reserved memory for training % of max memory = 0.0 %.


In [13]:
# model.save_pretrained("qwen-7b_lora_model")
# tokenizer.save_pretrained("qwen-7b_lora_model")


model.save_pretrained_merged("qwen-7b_lora_model_merged", tokenizer, )

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model-00001-of-00004.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Unsloth: Merging weights into 16bit:   0%|          | 0/4 [00:00<?, ?it/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.88G [00:00<?, ?B/s]

  2025-09-28T07:05:47.183422Z  WARN  Reqwest(reqwest::Error { kind: Request, url: "https://cas-server.xethub.hf.co/reconstructions/8e3cec323468944e3b5a2ca853b0dd73285468b09e6f568845bc72f3b165d820", source: hyper_util::client::legacy::Error(Connect, Custom { kind: Other, error: Os { code: 110, kind: TimedOut, message: "Connection timed out" } }) }). Retrying...
    at /home/runner/work/xet-core/xet-core/cas_client/src/http_client.rs:233

  2025-09-28T07:05:47.183482Z  WARN  Retry attempt #0. Sleeping 107.79038ms before the next attempt
    at /root/.cargo/registry/src/index.crates.io-1949cf8c6b5b557f/reqwest-retry-0.7.0/src/middleware.rs:171



...

Unsloth: Merging weights into 16bit:  50%|█████     | 2/4 [12:20<12:22, 371.12s/it]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.33G [00:00<?, ?B/s]

Unsloth: Merging weights into 16bit:  75%|███████▌  | 3/4 [19:29<06:37, 397.51s/it]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.09G [00:00<?, ?B/s]

Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [20:43<00:00, 310.97s/it]


In [17]:
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

question="一个患有急性阑尾炎的病人已经发病5天，腹痛稍有减轻但仍然发热，在体检时发现右下腹有压痛的包块，此时应如何处理？", # Question
inputs = tokenizer([inference_prompt.format(question, "")], return_tensors="pt").to("cuda")

outputs = model.generate(
    input_ids=inputs.input_ids,
    attention_mask=inputs.attention_mask,
    max_new_tokens=1000,
    use_cache=True,
)

In [18]:
output = tokenizer.batch_decode(outputs, skip_special_tokens=True)
print(output[0].split("### Response:")[1])


<think>
这个病人已经病了五天，腹痛感觉稍微好了一点，但是还是有点儿发热呢。哦，右下腹有个压痛的包块，这有点儿不对劲。我得想想，压痛包块可能是什么情况呢。

首先，我得考虑这个包块是炎症还是别的什么。急性阑尾炎本来就是个炎症的问题，所以包块是正常的炎症表现。不过，发热的话，这可能不是阑尾炎，因为阑尾炎一般不会这么高烧。

然后，我想到可能是个结核，结核病也会有包块，但结核一般不会这么热，而且结核的压痛包块一般会很肿。这个病人虽然发热，但没有那么高，所以可能不太是结核。

再想想，还有个情况，就是急性阑尾炎的时候，可能有包块，但通常不会这么热。所以，这个包块可能不是阑尾炎。

还有，我想到别的可能，比如急性胰腺炎，这种病也容易有包块，而且一般会有高烧。哦，这个病人虽然有发热，但压痛包块可能就是胰腺炎的表现。

不过，急性胰腺炎的包块一般会更肿，不太像这个病人现在的情况。所以，可能不是急性胰腺炎。

那还有其他可能吗？比如急性胆囊炎，这个情况不太常见，而且通常不会有包块。

再想想，有没有可能这个包块是别的器官的问题呢？比如胆囊炎，但不太常见。

或者，可能这是一个急性胃炎，但是胃炎一般不会有这么大的包块。

等等，有没有可能这个包块是急性胰腺炎的表现呢？急性胰腺炎通常会有高烧，而且会有腹痛和包块，但包块通常会更肿，不太像这个病人现在的情况。

哦，还有，我记得急性胰腺炎通常会有明显的腹痛和包块，而这个病人虽然腹痛减轻了，但还有发热，这可能不太符合急性胰腺炎。

再想想，还有急性胰腺炎的其他表现，比如急性胰腺炎通常会有明显的胰腺肿胀，而这个病人可能没有那么明显的肿胀。

那么，现在看起来，这个包块不太可能是急性胰腺炎的表现。

那还有其他可能吗？比如急性胰腺炎，但这个病人可能不是急性胰腺炎。

那么，现在我觉得这个包块不太可能是急性胰腺炎的表现，可能还是阑尾炎。

但，再仔细想想，急性阑尾炎一般不会有这么大的包块，而且通常不会有这么高的烧。

哦，等等，我好像有点儿混淆了。急性阑尾炎一般不会有这么大的包块，而且一般不会有这么高的烧。所以，这个包块可能不是急性阑尾炎的表现。

那我再想想，有没有其他可能，比如急性胰腺炎，但这个病人不是急性胰腺炎。

或者，可能这是一个急性胆囊炎，但胆囊炎一般不会有这么大的包块。

等等，这个包块可能是急性胰腺炎的表现，虽然有点儿不

In [19]:
def generate_response(question: str, model, tokenizer, inference_prompt: str, max_new_tokens: int = 1024) -> str:
    """
    使用指定的模型和分词器为给定的医学问题生成响应。

    Args:
        question (str): 需要模型回答的医学问题。
        model: 已加载的 Unsloth/Hugging Face 模型。
        tokenizer: 对应的分词器。
        inference_prompt (str): 用于格式化输入的 f-string 模板。
        max_new_tokens (int, optional): 生成响应的最大 token 数量。默认为 1024。

    Returns:
        str: 模型生成的响应文本，已去除 prompt 部分。
    """
    # 1. 使用模板格式化输入
    prompt = inference_prompt.format(
        question, # 填充问题
        "",       # 留空，让模型生成 CoT 和 Response
    )

    # 2. 将格式化后的 prompt 进行分词，并转移到 GPU
    inputs = tokenizer([prompt], return_tensors="pt").to(model.device)

    # 3. 使用模型生成输出
    # use_cache=True 用于加速解码过程
    outputs = model.generate(
        input_ids=inputs.input_ids,
        attention_mask=inputs.attention_mask,
        max_new_tokens=max_new_tokens,
        use_cache=True,
    )
    
    # 4. 将生成的 token 解码为文本
    # skip_special_tokens=True 会移除像 EOS_TOKEN 这样的特殊标记
    decoded_output = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

    # 5. 切分字符串，只返回 "### Response:" 之后的部分
    # 使用 .split() 分割并获取响应内容，.strip() 用于去除可能存在的前后空白字符
    response_part = decoded_output.split("### Response:")
    if len(response_part) > 1:
        return response_part[1].strip()
    else:
        # 如果模型没有生成 "### Response:" 标记，则返回整个生成内容以供调试
        return decoded_output

In [16]:
my_question = "对于一名60岁男性患者，出现右侧胸疼并在X线检查中显示右侧肋膈角消失，诊断为肺结核伴右侧胸腔积液，请问哪一项实验室检查对了解胸水的性质更有帮助？"

response = generate_response(my_question, model, tokenizer, inference_prompt)
print("==================== 模型回答 ====================")
print(response)

==================== 模型回答 ====================
<think>
嗯，这位60岁的男性患者，右胸疼痛，而且在X线检查中发现右侧肋膈角消失。这让我想到肺结核，而且还有右侧胸腔积液。嗯，肺结核患者常常会有胸水，特别是像这样的情况，可能是因为结核病引起的胸水。嗯，现在问题是要找一个能够帮助了解胸水性质的实验室检查。

哦，胸水的性质，这听起来像是要了解它的成分。我想，这可能需要一些血液检测。比如，血常规，它能显示红细胞、白细胞、血小板这些，也能看到一些微生物，比如细菌、病毒，这些可能会影响胸水的性质。

哦，对了，还有抗酸血症的检测。这会显示血液中酸的含量，这跟胸水中的酸性物质有关。另外，抗酸血症还能显示一些细菌，比如肺炎链球菌，这些可能会对胸水的性质有影响。

嗯，这样看来，血常规和抗酸血症检测都可能帮助了解胸水的成分。嗯，我觉得抗酸血症检测可能更直接，因为它直接显示血液中的酸性物质和细菌。这样，我们就能更清楚地了解胸水中的微生物情况。

嗯，嗯，好，那我倾向于选择抗酸血症的检测，因为它直接关联到胸水中的酸性物质和细菌，这能帮助我们更好地了解胸水的性质。
</think>
在60岁男性患者出现右胸疼痛并伴有右侧肋膈角消失的情况下，考虑到患者可能是肺结核伴右侧胸腔积液的典型表现，为了了解胸水的性质，最合适的实验室检查是抗酸血症的检测。抗酸血症检测能够显示血液中酸的含量以及其中的细菌，如肺炎链球菌，这些信息对于分析胸水的成分和性质非常关键。因此，抗酸血症检测是了解胸水性质的更直接和有效的方法。


In [20]:
my_question = "对于一名 28 岁的男性患者，工作是程序员，常年熬夜，最近突然感觉头晕目眩，甚至有点恶心。请问有可能是什么疾病？"

response = generate_response(my_question, model, tokenizer, inference_prompt, 512)
print("==================== 模型回答 ====================")
print(response)

==================== 模型回答 ====================
<think>
这位28岁的男性程序员，每天熬夜工作，最近突然感觉头晕目眩，甚至有点恶心，看起来是不太舒服啊。让我仔细想想，这可能是什么原因呢。

首先，他每天晚上熬夜工作，长时间盯着电脑，这对眼睛还是有点问题的。长时间用眼可能导致眼睛疲劳，甚至出现一些不适症状。但是，他最近突然出现头晕目眩，这可能跟眼疲劳有关，也可能跟身体的其他状况有关。

然后，他最近感觉有点恶心，这可能是因为身体内部有一些不平衡，或者是由于身体的某些器官出现问题。比如，他的头部和身体的协调可能出现了问题，导致他感到不适。

再想想，他最近的这些症状，是否和他最近的工作习惯有关呢？他工作时间长，可能需要做一些需要身体协调的活动，比如打游戏或者编程。这些活动可能会影响他的身体平衡，导致他出现这些症状。

嗯，可能他最近身体有些疲劳，或者有轻微的健康问题。但如果是这样的话，应该不会出现明显的头晕和恶心。所以，或许他的身体内部有一些小问题，需要进一步检查。

也许他最近的这些症状，和他身体的某些器官有关。比如，他的头部和身体协调能力可能出现了问题，导致他感到头晕和恶心。这可能与他的身体协调能力有关，比如他的神经系统的协调能力。

或者，他的身体内部可能有一些微小的不平衡，比如他的头部和身体协调能力不一致，导致他感到不适。这种情况下，他可能需要做一些身体协调的活动，比如做一些轻微的运动，以缓解这种不平衡。

不过，总的来说，他最近的这些症状，可能和他长时间的使用眼睛有关，或者他身体内部的某些协调问题有关。要确定具体的原因，最好还是做一些进一步的检查，比如眼疲劳检测或者身体协调能力的测试。

总之，这位28岁的程序员最近出现头晕和恶心，可能是由于长时间使用眼睛或者身体协调能力的问题。为了确定具体的原因，建议他做一些进一步的检查，以排除其他可能性。
</think>
这位28岁的程序员最近出现头晕和恶心，可能是由于长时间使用眼睛导致的眼疲劳，或者身体协调能力的问题。长时间使用眼睛，尤其是盯着屏幕工作，可能会导致眼睛疲劳，进而引发头晕和恶心的症状。此外，身体协调能力的问题也可能导致身体内部的不平衡，进而引发这些症状。

为了进一步确定具体的原因，建议他做一些进一步的检查，比如眼疲劳检测或身体协调能力测试
